In [7]:
## layer 1, understand strides and kernel_size concepts, we do not freeze the layers i think? maybe if there is overfitting we can freeze resnet for less training parameters
# TODO: Ask about copying codes from documentation
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import os
# https://flypix.ai/blog/image-recognition-algorithms/ why I choosed only first few layers for feature extraction, upto middle layers are sufficient to provide information about eyes
# In higher layers of the network, detailed pixel information is lost whilethe high level content of the image is preserved. Clear Explanation is here(https://ai.stackexchange.com/questions/30038/why-do-we-lose-detail-of-an-image-as-we-go-deeper-into-a-convnet)
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super(ResNetFeatureExtractor, self).__init__()
        resnet = models.resnet18(pretrained=True)
        print(resnet)

        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-3])
        
        
        self.feature_extractor[-1][0].conv1.stride = (1, 1) # normally resnet has stride = 2 in layer 3 this will downsample from 8x8 to 4x4. This is to avoid this. More spatil features are preserved
        self.feature_extractor[-1][0].downsample[0].stride = (1, 1)

        self.reduce_channels = nn.Conv2d(256, 128, kernel_size=1) # kernel_size 1 only changes the number of channels and doesn't mess with the spatial size

    def forward(self, x):
        x = self.feature_extractor(x)  
        x = self.reduce_channels(x)   
        return x

feature_extractor = ResNetFeatureExtractor()
feature_extractor.eval()  

transform = transforms.Compose([ # pre processing the images to match the resnet's training statistics
    transforms.Resize((128, 128)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalization of the pixel values
])

def extract_features(image_path):
    img = Image.open(image_path).convert("RGB") 
    img = transform(img).unsqueeze(0) # apply the transformations, batch size is set to 1
    # with torch.no_grad():
    features = feature_extractor(img)  
    
    return features

dataset_path = "./Test Dataset/Output/"

left_eye_img = os.path.join(dataset_path, "webcam_r/left_eye/left_eye_0000.jpg")
right_eye_img = os.path.join(dataset_path, "webcam_r/right_eye/right_eye_0000.jpg")
face_img = os.path.join(dataset_path, "webcam_r/face/face_0000.jpg")


left_eye_features = extract_features(left_eye_img)  
right_eye_features = extract_features(right_eye_img)  
face_features = extract_features(face_img) 


print(left_eye_features.shape)
print(right_eye_features.shape)
print(face_features.shape)

c:\Users\A98026663\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\A98026663\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [3]:
import torch 


# input of layer 2
Leye = torch.rand(128, 8, 8)
print(Leye)

Reye = torch.rand(128, 8, 8)
print(Reye)

FaceData = torch.rand(32, 8, 8)
print(FaceData)

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        

        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        self.gn = torch.nn.GroupNorm(3, 6)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?

        # layer 3:.......................
    
    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((torch.tensor(left_eye), torch.tensor(right_eye), torch.tensor(face)), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        return out

model = TinyModel()
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)


tensor([[[6.0070e-01, 6.2904e-01, 8.0323e-02,  ..., 1.2767e-01,
          5.6666e-01, 6.3905e-01],
         [7.1470e-01, 6.2460e-01, 2.3516e-01,  ..., 2.6767e-01,
          6.8786e-01, 7.6928e-01],
         [8.7115e-02, 4.9256e-01, 2.4544e-01,  ..., 7.4388e-01,
          1.7701e-01, 8.9320e-01],
         ...,
         [5.4316e-01, 8.7439e-01, 2.8298e-01,  ..., 2.4509e-01,
          9.8691e-01, 7.4261e-01],
         [9.2761e-01, 5.6528e-01, 9.5660e-01,  ..., 8.6643e-01,
          9.0488e-01, 8.3185e-01],
         [2.2024e-04, 2.0880e-01, 3.4194e-01,  ..., 8.5111e-01,
          6.5264e-01, 9.6603e-02]],

        [[9.1129e-01, 1.0415e-01, 1.4733e-01,  ..., 1.5724e-01,
          2.8314e-01, 5.8653e-01],
         [5.9759e-01, 1.1552e-01, 3.5343e-01,  ..., 9.2179e-01,
          6.4653e-01, 2.6380e-01],
         [4.8583e-01, 1.7367e-01, 6.5717e-01,  ..., 9.8532e-01,
          6.3170e-01, 1.7072e-01],
         ...,
         [7.9325e-01, 4.2677e-01, 4.2708e-02,  ..., 1.6591e-01,
          6.522

C:\Users\A98026663\AppData\Local\Temp\ipykernel_24536\1169748882.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  concate = torch.cat((torch.tensor(left_eye), torch.tensor(right_eye), torch.tensor(face)), 1)  # dim = 0 or 1?  only channel dim changes?


RuntimeError: Expected weight to be a vector of size equal to the number of channels in input, but got weight of shape [6] and input of shape [128, 24, 8]

In [ ]:
## If there is overfitting we can switch to 2D Convolutional network, https://github.com/swook/EVE/blob/master/DATASET.md for dataset information
inp = torch.rand(1, 384, 1, 1)
print(inp)

class GazePrediction(nn.Module):
    def __init__(self):
        super(GazePrediction, self).__init__()
        self.fc = nn.Linear(384, 3)  
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  
        x = self.fc(x) 
        return x


gaze_model = GazePrediction()

gaze_vector = gaze_model(inp)




tensor([[[[5.6424e-01]],

         [[6.7118e-01]],

         [[1.3195e-01]],

         [[1.5159e-02]],

         [[5.5969e-02]],

         [[3.2089e-01]],

         [[9.3360e-01]],

         [[1.4936e-01]],

         [[2.5786e-01]],

         [[2.1865e-01]],

         [[1.7244e-01]],

         [[4.9350e-01]],

         [[6.9637e-01]],

         [[8.9512e-02]],

         [[3.5183e-01]],

         [[5.6021e-01]],

         [[5.6474e-01]],

         [[7.6940e-01]],

         [[2.9841e-01]],

         [[9.3738e-01]],

         [[1.1769e-01]],

         [[5.4468e-01]],

         [[6.7953e-01]],

         [[2.9714e-02]],

         [[6.7954e-01]],

         [[7.3950e-02]],

         [[5.0909e-01]],

         [[1.9736e-01]],

         [[4.7741e-01]],

         [[8.9459e-01]],

         [[9.1767e-02]],

         [[2.4295e-01]],

         [[1.2453e-01]],

         [[5.3793e-01]],

         [[6.9565e-01]],

         [[9.5778e-01]],

         [[2.8232e-02]],

         [[3.5357e-01]],

         [[7